## All necessary imports - do it here

In [3]:
import numpy as np
from pathlib import Path
import gym_electric_motor as gem
from gym_electric_motor.reference_generators import ConstReferenceGenerator
from gym_electric_motor.visualization import MotorDashboard
from gym_electric_motor.reward_functions.weighted_sum_of_errors import WeightedSumOfErrors
from gym_electric_motor.core import Callback
from gym_electric_motor.envs.motors import ActionType, ControlType, Motor, MotorType
from gym_electric_motor.physical_systems.solvers import EulerSolver
from gymnasium.spaces import Discrete, Box, Tuple
from gymnasium.wrappers import FlattenObservation, TimeLimit
from gymnasium import ObservationWrapper
from gym_electric_motor.physical_systems import ConstantSpeedLoad
import gymnasium as gym

## 1. Feature Engineering

wrap the env returned by gym.make to have the desired feature vector , call the reset to pass the varying control cycle time
Flattenobsv --> FeatureWrapper (with custom reset and New observation vector) --> gym env


#### Agent work flow
Reset(from the wrapped env --> base env) --> Step --> Reset --> ....

In [4]:
class FeatureWrapper(ObservationWrapper):
  
    def __init__(self, env):
        """
        Changes the observation space to fit the new features
        
        Args:
            env(GEM env): GEM environment to wrap
        """
        super(FeatureWrapper, self).__init__(env)
        self.tau_index = [0]
        self.tau_max = 100e-6

        '''
         feature vector = [omega_me, i_d, i_q, u_d, u_q, k*cos(eps), k*sin(eps), 2*i_s -1, varying_cycle_Time, Torque()from reference, last action]
         k to dampen the angle information
         i_s = np.sqrt(i_d ** 2 + i_q ** 2)
         from physical system --> (mechanical_state, [torque], i_abc,  i_dq,   u_in,   u_dq,  [eps], u_sup)
                                     0                1       2,3,4   5,6   7,8,9   10,11    12     ...
        '''
        new_low = np.concatenate(([self.env.observation_space.spaces[0].low[0]],
                                  self.env.observation_space.spaces[0].low[5:7],# currents in dq
                                  self.env.observation_space.spaces[0].low[10:12],# voltages in dq
                                  [-1, -1],# angles in cos, sin 
                                  [-1],#total stator current, i_s
                                  [0],#varying_cycle_Time
                                  [-1],#to accomodate the torque ref
                                  [-1, -1, 1]))#last action
        new_high = np.concatenate(([self.env.observation_space.spaces[0].high[0]],
                                  self.env.observation_space.spaces[0].high[5:7],
                                  self.env.observation_space.spaces[0].high[10:12],
                                  [1, 1],
                                  [1],#total stator current, i_s
                                  [1],#varying_cycle_Time
                                  [1],#to accomodate the torque ref
                                  [1, 1, 1]))#last action

        self.observation_space = Box(new_low, new_high) # how to add the ref torque 

    def T_s_selector(self,array):
        value = array[self.tau_index[0]]
        return value
    
    def reset(self, **kwargs):
        varying_T_s_Array = [25e-6, 50e-6]
        observation, info = self.env.reset(options={"varying_T_s": self.T_s_selector(varying_T_s_Array)})

        state = observation[0]
        ref_torque = observation[1][0]#high and low are the same

        normalized_cycle_tme = self.T_s_selector(varying_T_s_Array) / self.tau_max
        self.tau_index[0] = 1 - self.tau_index[0]
        
        omega_me = state[0]
        # what should be the u_dq given out to the agent..how to derive this from the action.
        u_dq = [0,0] # ???

        i_sd = state[5]
        i_sq = state[6]
        i_s = np.sqrt(i_sd ** 2 + i_sq ** 2)
    
    
        k = 0.1 
        eps = state[12]
        angles = [k * np.cos(eps * np.pi), k * np.sin(eps * np.pi)]

        action_placeholder = [0, 0, 0]# will be replaced from the outer wrapper

        feature_vector = np.concatenate(([omega_me],
                                           [i_sd, i_sq],
                                           u_dq,
                                           angles,
                                           [2 * i_s -1],
                                           [normalized_cycle_tme],
                                           [ref_torque],
                                           action_placeholder))

        return feature_vector, info
    
    def step(self, action):
        (state, reference), reward, terminated, truncated, info = self.env.step(action)

        normalized_cycle_tme = self.env.get_wrapper_attr('_physical_system')._tau / self.tau_max
        omega_me = state[0]
        ref_torque = reference[0]

        #u_abc = self.subactions[action]
        #u_dq = self.env.physical_system.abc_to_dq_space(u_abc, epsilon_el=eps * np.pi)
        u_d = state[10]
        u_q = state[11]
        u_dq = [u_d, u_q] #but is it right ?because the effect of action at the current step will only be visible in the upcoming steps

        i_sd = state[5]
        i_sq = state[6]
        i_s = np.sqrt(i_sd ** 2 + i_sq ** 2)
    

        k = 0.1 
        eps = state[12]
        angles = [k * np.cos(eps * np.pi), k * np.sin(eps * np.pi)]

        action_placeholder = [0, 0, 0] # will be replaced from the outer wrapper

        feature_vector = np.concatenate(([omega_me],
                                           [i_sd, i_sq],
                                           u_dq,
                                           angles,
                                           [2 * i_s -1],
                                           [normalized_cycle_tme],
                                           [ref_torque],
                                           action_placeholder))

        #return (observable_state , reference), reward, terminated, truncated
        return feature_vector, reward, terminated, truncated, info
        

In [5]:

class LastActionWrapper(gym.Wrapper):
    def __init__(self, env):
        super(LastActionWrapper, self).__init__(env)
        state_space = self.env.observation_space

        self.subactions = -np.power(-1, self.env.get_wrapper_attr('physical_system')._converter._subactions)
    
        self.observation_space = state_space

    def reset(self, **kwargs):
        observation, info = self.env.reset(**kwargs)
        feature_vector_without_lastaction = observation[:10]
        self.last_action = self.subactions[0]
        self.last_reward = np.nan
        return np.concatenate((feature_vector_without_lastaction, self.last_action)), info

    # put action to the last action
    # added safe guarding and individual reward function here.
    def step(self, action):

        observation, reward, terminated, truncated, info = self.env.step(action)
        feature_vector_without_lastaction = observation[:10]
        self.last_action = self.subactions[action]
        self.last_reward = reward

        return np.concatenate((feature_vector_without_lastaction, self.last_action)), reward, terminated, truncated, info

### Define env parameters and max episode step

In [6]:
torque_ref_generator = ConstReferenceGenerator(reference_state='torque', reference_value=np.random.uniform(-1, 1))

motor_parameter = dict(p=3,            # [p] = 1, nb of pole pairs
                       r_s=17.932e-3,  # [r_s] = Ohm, stator resistance
                       l_d=0.37e-3,    # [l_d] = H, d-axis inductance
                       l_q=1.2e-3,     # [l_q] = H, q-axis inductance
                       psi_p=65.65e-3, # [psi_p] = Vs, magnetic flux of the permanent magnet
                       )  # BRUSA

u_sup = 350
nominal_values=dict(omega=12000*2*np.pi/60,
                    i=240,
                    u=u_sup)

limit_values=nominal_values.copy()
limit_values["i"] = 270
limit_values["torque"] = 200

sampling_time = 50e-6 # dummy value only used to create the env..will be over written.

# define maximal episode steps
max_eps_steps = 10000

## 2. Create the env

In [7]:
base_env = gem.make("Finite-TC-PMSM-v0",
                   motor = dict(
                       motor_parameter=motor_parameter,
                       limit_values=limit_values,
                       nominal_values=nominal_values,
                   ),
                   supply=dict(u_nominal=u_sup),
                   load = ConstantSpeedLoad(omega_fixed=200),
                   tau=sampling_time,
                   reward_function=WeightedSumOfErrors(reward_weights={'torque': 1},  # but the reward distribution will be overwritten
                                                              gamma=0.868), # by means of the defined wrapper function
                   reference_generator=torque_ref_generator,
                   )

c:\Users\RANIL THOMAS\anaconda3\envs\varyingFreq_WS25\lib\site-packages\gymnasium\utils\passive_env_checker.py:32: UserWarning: WARN: A Box observation space maximum and minimum values are equal. Actual equal coordinates: [(0,)]
  logger.warn(


### Wrap the base env to present it to the learning agent


In [8]:
env = LastActionWrapper(FeatureWrapper(base_env))

#### Testing

In [9]:
terminated = True
for _ in range(100):
   if terminated:
     obs = env.reset()
   obs, reward, terminated, truncated, info = env.step(1)
   

c:\Users\RANIL THOMAS\anaconda3\envs\varyingFreq_WS25\lib\site-packages\gymnasium\utils\passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## 3. Moving to Training

In [10]:
import matplotlib.pyplot as plt
from stable_baselines3 import DQN
from stable_baselines3.dqn import MlpPolicy
from pathlib import Path

### Agent parameters

In [11]:
buffer_size = 200000 #number of old obsersation steps saved
learning_starts = 10000 # memory warmup
train_freq = 1 # prediction network gets an update each train_freq's step
batch_size = 25 # mini batch size drawn at each update step
policy_kwargs = {
        'net_arch': [64,64] # hidden layer size of MLP
        }
exploration_fraction = 0.1 # Fraction of training steps the epsilon decays 
target_update_interval = 1000 # Target network gets updated each target_update_interval's step
gamma = 0.99
verbose = 1 # verbosity of stable-basline's prints

tau = 1e-3
simulation_time = 5 # seconds
nb_steps = int(simulation_time // tau)

### Training

In [12]:
model = DQN(MlpPolicy, env, buffer_size=buffer_size, learning_starts=learning_starts ,train_freq=train_freq, 
            batch_size=batch_size, gamma=gamma, policy_kwargs=policy_kwargs, 
            exploration_fraction=exploration_fraction, target_update_interval=target_update_interval,
            verbose=verbose)
model.learn(total_timesteps=1000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 120      |
|    ep_rew_mean      | -20.6    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 3798     |
|    time_elapsed     | 0        |
|    total_timesteps  | 479      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 113      |
|    ep_rew_mean      | -18.7    |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 3430     |
|    time_elapsed     | 0        |
|    total_timesteps  | 903      |
----------------------------------
